# Identify Locations using Open Data Classifications

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import gc

In [ ]:
active_pc = gpd.read_parquet('/Users/pedrickr/Documents/ADR/Digital_Inclusion/active_pc_centroids_rural_flag.parquet')
rural_pc_centroids = active_pc[active_pc['is_rural'] == 1]

In [ ]:
cols = [
    'CompanyName',
    ' CompanyNumber',
    'SICCode.SicText_1',
    'RegAddress.PostCode',
    'CompanyStatus'
]

# Load only needed columns
ch = pd.read_csv(
    '/Users/pedrickr/Documents/ADR/Connectivity/BasicCompanyDataAsOneFile-2026-02-01.csv',
    usecols=cols,
    low_memory=False
)

# Rename columns
ch_df = ch.rename(columns={
    'CompanyName': 'companyName',
    ' CompanyNumber': 'CompanyNumber',
    'SICCode.SicText_1': 'sic_raw',
    'RegAddress.PostCode': 'postcode_clean',
    'CompanyStatus': 'status'
}).copy()

# Clean postcode
ch_df["postcode_clean"] = ch_df["postcode_clean"].str.replace(" ", "", regex=False)

# Extract SIC code + description (single pass)
ch_df[["sic_code", "sic_description"]] = ch_df["sic_raw"].str.extract(
    r"^\s*(\d{5})\s*-\s*(.*)$"
)

# Free memory
del ch
gc.collect()

# Filter farming companies early
farm_prefixes = ("011", "012", "013", "014", "015")

ch_farming = ch_df[
    ch_df["sic_code"].str.startswith(farm_prefixes, na=False)
]

# Counts
count_ch_no = ch_df["CompanyNumber"].nunique()
ch_farming_count = ch_farming["CompanyNumber"].nunique()

print(f"Total companies: {count_ch_no}")
print(f"Farming companies: {ch_farming_count}")
print(f"% farming: {round((ch_farming_count / count_ch_no) * 100, 3)}")

# Postcode counts
unique_pc_count = ch_farming["postcode_clean"].nunique()
print(f"Unique farming postcodes: {unique_pc_count}")

# Join with rural postcode centroids
ch_farm_pc_rural = rural_pc_centroids.merge(
    ch_farming,
    on="postcode_clean",
    how="inner"
)

rural_pc_count = ch_farm_pc_rural["postcode_clean"].nunique()

print(
    "Unique rural farming postcodes: "
    + str(rural_pc_count)
)

ch_farms_list = list(ch_farm_pc_rural.postcode_clean.unique())

len(ch_farm_pc_rural.postcode_clean.unique())


# Open Street Map

In [ ]:
# Note: Ireland and NI are joined in the GeoFabrik OSM file, so we will have to clip this out later.
osm_wales  = gpd.read_file('/Users/pedrickr/Documents/ADR/Connectivity/wales-260405-free/gis_osm_landuse_a_free_1.shp')
osm_england = gpd.read_file('/Users/pedrickr/Documents/ADR/Connectivity/england-260405-free/gis_osm_landuse_a_free_1.shp')
osm_scotland = gpd.read_file('/Users/pedrickr/Documents/ADR/Connectivity/scotland-260405-free/gis_osm_landuse_a_free_1.shp')
osm_ireland_ni = gpd.read_file('/Users/pedrickr/Documents/ADR/Connectivity/ireland-and-northern-ireland-260420-free/gis_osm_landuse_a_free_1.shp')

osm_gb = pd.concat([osm_wales, osm_england, osm_scotland, osm_ireland_ni], axis=0, ignore_index=True)

del osm_wales, osm_england, osm_scotland, osm_ireland_ni
gc.collect()

osm_farmyard = osm_gb[osm_gb['fclass'] == 'farmyard']
osm_farmyard = osm_farmyard.to_crs('EPSG:27700')

In [ ]:
import numpy as np
from scipy.spatial import cKDTree

# Ensure CRS
osm_farmyard = osm_farmyard.to_crs("EPSG:27700")
rural_pc_centroids = rural_pc_centroids.to_crs("EPSG:27700")

# Coordinates (fast)
pc_coords = np.column_stack((
    rural_pc_centroids.geometry.x,
    rural_pc_centroids.geometry.y
))

farm_points = osm_farmyard.geometry.representative_point()

farm_coords = np.column_stack((
    farm_points.x,
    farm_points.y
))

# Build tree
tree = cKDTree(pc_coords)

# Query nearest neighbour
distances, indices = tree.query(
    farm_coords,
    k=1,
    distance_upper_bound=5000
)

# Mask valid matches
valid = distances != float("inf")

# Map back safely
osm_farm_pc = pd.DataFrame({
    "farm_id": osm_farmyard.index[valid],
    "postcode_clean": rural_pc_centroids.iloc[indices[valid]]["postcode_clean"].values,
    "distance": distances[valid]
    
})

osm_farm_list = osm_farm_pc.postcode_clean.unique()

# CH + OSM

In [ ]:
farm_pc_list = list(set(ch_farms_list) | set(osm_farm_list))

len(farm_pc_list)

In [ ]:
farm_pc_centroids = rural_pc_centroids[rural_pc_centroids['postcode_clean'].isin(farm_pc_list)]
farm_pc_centroids